# Module 9: Final Campus-Event Model

Implement the M8 blueprint. Validation chooses the model; the final test set is used once.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Load+the+cleaned+dataset+and+the+%22blueprint%22+%28the+plan%29+made+back+in+Module+8.%0Aimport+json%0Afrom+pathlib+import+Path%0Aimport+pandas+as+pd%0A%0Adf%3Dpd.read_csv%28%27campus_events_clean.csv%27%29%0A%23+Use+the+final+blueprint+if+it+exists%2C+otherwise+fall+back+to+the+checkpoint+version.%0Abp_path%3DPath%28%27m8_model_blueprint.json%27%29+if+Path%28%27m8_model_blueprint.json%27%29.exists%28%29+else+Path%28%27m8_model_blueprint_checkpoint.json%27%29%0Ablueprint%3Djson.loads%28bp_path.read_text%28%29%29%0Aprint%28%27Blueprint%3A%27%2Cbp_path.name%2C%27Dataset%3A%27%2Cdf.shape%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Load the cleaned dataset and the "blueprint" (the plan) made back in Module 8.
import json
from pathlib import Path
import pandas as pd

df=pd.read_csv('campus_events_clean.csv')
# Use the final blueprint if it exists, otherwise fall back to the checkpoint version.
bp_path=Path('m8_model_blueprint.json') if Path('m8_model_blueprint.json').exists() else Path('m8_model_blueprint_checkpoint.json')
blueprint=json.loads(bp_path.read_text())
print('Blueprint:',bp_path.name,'Dataset:',df.shape)

## 1. Confirm the M8 blueprint
Record any revision before modeling. Do not quietly add rejected or post-event features.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Atarget%3Dblueprint%5B%27target%27%5D%0Afeatures%3Dblueprint%5B%27accepted_features%27%5D%0A%23+These+columns+would+%22cheat%22+%28they%27re+only+known+after+the+event+happens%29%2C+so+they+must+never+be+used+as+inputs.%0Ablocked%3D%7B%27event_id%27%2C%27actual_weather%27%2C%27actual_attendance%27%2C%27attendance_rate%27%2C%27high_turnout%27%7D%0Aassert+not+blocked.intersection%28features%29%2C+f%27Blocked+feature+selected%3A+%7Bblocked.intersection%28features%29%7D%27%0Aprint%28%27Target%3A%27%2Ctarget%2C%27Features%3A%27%2Cfeatures%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
target=blueprint['target']
features=blueprint['accepted_features']
# These columns would "cheat" (they're only known after the event happens), so they must never be used as inputs.
blocked={'event_id','actual_weather','actual_attendance','attendance_rate','high_turnout'}
assert not blocked.intersection(features), f'Blocked feature selected: {blocked.intersection(features)}'
print('Target:',target,'Features:',features)

## 2. Build a preprocessing pipeline
Numeric columns pass through; categorical columns are one-hot encoded. Fit preprocessing only on training data.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Afrom+sklearn.model_selection+import+train_test_split%0Afrom+sklearn.compose+import+ColumnTransformer%0Afrom+sklearn.pipeline+import+Pipeline%0Afrom+sklearn.preprocessing+import+OneHotEncoder%2C+StandardScaler%0Afrom+sklearn.impute+import+SimpleImputer%0Afrom+sklearn.tree+import+DecisionTreeClassifier%0Afrom+sklearn.linear_model+import+LogisticRegression%0Afrom+sklearn.metrics+import+accuracy_score%2C+precision_score%2C+recall_score%2C+confusion_matrix%2C+classification_report%0A%23+X+%3D+the+input+columns+%28features%29%2C+y+%3D+the+answer+we+want+to+predict+%28turned+into+0%2F1+instead+of+no%2Fyes%29.%0AX%3Ddf%5Bfeatures%5D.copy%28%29%3B+y%3Ddf%5Btarget%5D.map%28%7B%27no%27%3A0%2C%27yes%27%3A1%7D%29%0A%23+Split+the+data+twice%3A+first+carve+out+a+%22test%22+set+we+won%27t+touch+until+the+very+end%2C%0A%23+then+split+the+rest+into+%22train%22+%28to+fit+the+model%29+and+%22validation%22+%28to+compare+models%29.%0AX_build%2CX_test%2Cy_build%2Cy_test%3Dtrain_test_split%28X%2Cy%2Ctest_size%3D.20%2Crandom_state%3D42%2Cstratify%3Dy%29%0AX_train%2CX_val%2Cy_train%2Cy_val%3Dtrain_test_split%28X_build%2Cy_build%2Ctest_size%3D.25%2Crandom_state%3D42%2Cstratify%3Dy_build%29%0A%23+Separate+numeric+columns+%28like+counts%29+from+categorical+columns+%28like+text+labels%29%2C+since+they+need+different+handling.%0Anum%3DX.select_dtypes%28include%3D%27number%27%29.columns.tolist%28%29%3B+cat%3D%5Bc+for+c+in+X.columns+if+c+not+in+num%5D%0A%23+For+numbers%3A+fill+in+missing+values+with+the+median%2C+then+scale+them.%0A%23+For+categories%3A+fill+in+missing+values+with+the+most+common+value%2C+then+one-hot+encode+%28turn+labels+into+0%2F1+columns%29.%0Apre%3DColumnTransformer%28%5B%28%27num%27%2CPipeline%28%5B%28%27imp%27%2CSimpleImputer%28strategy%3D%27median%27%29%29%2C%28%27scale%27%2CStandardScaler%28%29%29%5D%29%2Cnum%29%2C%28%27cat%27%2CPipeline%28%5B%28%27imp%27%2CSimpleImputer%28strategy%3D%27most_frequent%27%29%29%2C%28%27onehot%27%2COneHotEncoder%28handle_unknown%3D%27ignore%27%29%29%5D%29%2Ccat%29%5D%29%0Aprint%28len%28X_train%29%2Clen%28X_val%29%2Clen%28X_test%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
# X = the input columns (features), y = the answer we want to predict (turned into 0/1 instead of no/yes).
X=df[features].copy(); y=df[target].map({'no':0,'yes':1})
# Split the data twice: first carve out a "test" set we won't touch until the very end,
# then split the rest into "train" (to fit the model) and "validation" (to compare models).
X_build,X_test,y_build,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_build,y_build,test_size=.25,random_state=42,stratify=y_build)
# Separate numeric columns (like counts) from categorical columns (like text labels), since they need different handling.
num=X.select_dtypes(include='number').columns.tolist(); cat=[c for c in X.columns if c not in num]
# For numbers: fill in missing values with the median, then scale them.
# For categories: fill in missing values with the most common value, then one-hot encode (turn labels into 0/1 columns).
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
print(len(X_train),len(X_val),len(X_test))

## 3. Establish the baseline

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Baseline+%3D+%22just+guess+the+most+common+answer+every+time.%22+Any+real+model+should+beat+this.%0Abaseline%3Dint%28y_train.mode%28%29.iloc%5B0%5D%29%3B+baseline_val%3Dfloat%28%28y_val%3D%3Dbaseline%29.mean%28%29%29%3B+print%28%27Validation+baseline%3A%27%2Cround%28baseline_val%2C3%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Baseline = "just guess the most common answer every time." Any real model should beat this.
baseline=int(y_train.mode().iloc[0]); baseline_val=float((y_val==baseline).mean()); print('Validation baseline:',round(baseline_val,3))

## 4. Compare candidates on validation data only

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Try+two+different+model+types+and+see+which+one+predicts+better.%0Acandidates%3D%7B%27decision_tree%27%3ADecisionTreeClassifier%28max_depth%3D4%2Cmin_samples_leaf%3D8%2Crandom_state%3D42%29%2C%27logistic_regression%27%3ALogisticRegression%28max_iter%3D1000%29%7D%0Avalidation%3D%5B%5D%3B+fitted%3D%7B%7D%0Afor+name%2Cmodel+in+candidates.items%28%29%3A%0A+%23+Combine+the+preprocessing+steps+with+the+model%2C+train+on+the+training+set%2C+then+predict+on+the+validation+set.%0A+pipe%3DPipeline%28%5B%28%27preprocess%27%2Cpre%29%2C%28%27model%27%2Cmodel%29%5D%29%3B+pipe.fit%28X_train%2Cy_train%29%3B+pred%3Dpipe.predict%28X_val%29%3B+fitted%5Bname%5D%3Dpipe%0A+%23+Score+each+model%3A+accuracy+%28overall+correctness%29%2C+precision+%28when+it+says+%22yes%22%2C+how+often+is+it+right%29%2C+recall+%28of+all+real+%22yes%22es%2C+how+many+did+it+catch%29.%0A+validation.append%28%7B%27model%27%3Aname%2C%27accuracy%27%3Aaccuracy_score%28y_val%2Cpred%29%2C%27precision%27%3Aprecision_score%28y_val%2Cpred%2Czero_division%3D0%29%2C%27recall%27%3Arecall_score%28y_val%2Cpred%2Czero_division%3D0%29%7D%29%0Avalidation_table%3Dpd.DataFrame%28validation%29%3B+validation_table"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Try two different model types and see which one predicts better.
candidates={'decision_tree':DecisionTreeClassifier(max_depth=4,min_samples_leaf=8,random_state=42),'logistic_regression':LogisticRegression(max_iter=1000)}
validation=[]; fitted={}
for name,model in candidates.items():
 # Combine the preprocessing steps with the model, train on the training set, then predict on the validation set.
 pipe=Pipeline([('preprocess',pre),('model',model)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_val); fitted[name]=pipe
 # Score each model: accuracy (overall correctness), precision (when it says "yes", how often is it right), recall (of all real "yes"es, how many did it catch).
 validation.append({'model':name,'accuracy':accuracy_score(y_val,pred),'precision':precision_score(y_val,pred,zero_division=0),'recall':recall_score(y_val,pred,zero_division=0)})
validation_table=pd.DataFrame(validation); validation_table

## 5. Lock the model, then refit on train + validation

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Pick+the+model+with+the+best+accuracy+%28recall+breaks+ties%29%2C+then+retrain+it+on+train%2Bvalidation+combined+for+the+strongest+final+version.%0Achosen_name%3Dvalidation_table.sort_values%28%5B%27accuracy%27%2C%27recall%27%5D%2Cascending%3DFalse%29.iloc%5B0%5D%5B%27model%27%5D%3B+chosen%3DPipeline%28%5B%28%27preprocess%27%2Cpre%29%2C%28%27model%27%2Ccandidates%5Bchosen_name%5D%29%5D%29%3B+chosen.fit%28X_build%2Cy_build%29%3B+print%28%27Locked%3A%27%2Cchosen_name%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Pick the model with the best accuracy (recall breaks ties), then retrain it on train+validation combined for the strongest final version.
chosen_name=validation_table.sort_values(['accuracy','recall'],ascending=False).iloc[0]['model']; chosen=Pipeline([('preprocess',pre),('model',candidates[chosen_name])]); chosen.fit(X_build,y_build); print('Locked:',chosen_name)

## 6. Use the final test set once

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+This+is+the+ONE+and+ONLY+time+we+use+the+test+set%2C+so+these+numbers+reflect+true%2C+unbiased+performance.%0Atest_pred%3Dchosen.predict%28X_test%29%3B+metrics%3D%7B%27accuracy%27%3Afloat%28accuracy_score%28y_test%2Ctest_pred%29%29%2C%27precision%27%3Afloat%28precision_score%28y_test%2Ctest_pred%2Czero_division%3D0%29%29%2C%27recall%27%3Afloat%28recall_score%28y_test%2Ctest_pred%2Czero_division%3D0%29%29%7D%3B+cm%3Dconfusion_matrix%28y_test%2Ctest_pred%2Clabels%3D%5B0%2C1%5D%29%3B+print%28metrics%29%3B+print%28pd.DataFrame%28cm%2Cindex%3D%5B%27actual+low%27%2C%27actual+high%27%5D%2Ccolumns%3D%5B%27predicted+low%27%2C%27predicted+high%27%5D%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# This is the ONE and ONLY time we use the test set, so these numbers reflect true, unbiased performance.
test_pred=chosen.predict(X_test); metrics={'accuracy':float(accuracy_score(y_test,test_pred)),'precision':float(precision_score(y_test,test_pred,zero_division=0)),'recall':float(recall_score(y_test,test_pred,zero_division=0))}; cm=confusion_matrix(y_test,test_pred,labels=[0,1]); print(metrics); print(pd.DataFrame(cm,index=['actual low','actual high'],columns=['predicted low','predicted high']))

## 7. Inspect mistakes and compare with the baseline

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Atest_results%3DX_test.copy%28%29%3B+test_results%5B%27actual%27%5D%3Dy_test%3B+test_results%5B%27predicted%27%5D%3Dtest_pred%0A%23+Grab+one+example+of+each+mistake+type%3A+a+false+positive+%28predicted+%22yes%22+but+actually+%22no%22%29...%0Afalse_positive%3Dtest_results%5B%28test_results.actual%3D%3D0%29%26%28test_results.predicted%3D%3D1%29%5D.head%281%29%0A%23+...and+a+false+negative+%28predicted+%22no%22+but+actually+%22yes%22%29.%0Afalse_negative%3Dtest_results%5B%28test_results.actual%3D%3D1%29%26%28test_results.predicted%3D%3D0%29%5D.head%281%29%0Aprint%28%27Final+majority+baseline%3A%27%2Cround%28float%28%28y_test%3D%3Dint%28y_build.mode%28%29.iloc%5B0%5D%29%29.mean%28%29%29%2C3%29%29%3B+display%28false_positive%29%3B+display%28false_negative%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
test_results=X_test.copy(); test_results['actual']=y_test; test_results['predicted']=test_pred
# Grab one example of each mistake type: a false positive (predicted "yes" but actually "no")...
false_positive=test_results[(test_results.actual==0)&(test_results.predicted==1)].head(1)
# ...and a false negative (predicted "no" but actually "yes").
false_negative=test_results[(test_results.actual==1)&(test_results.predicted==0)].head(1)
print('Final majority baseline:',round(float((y_test==int(y_build.mode().iloc[0])).mean()),3)); display(false_positive); display(false_negative)

## 8. Stress-test the model edges
Use three fictional cases: unfamiliar category, extreme promotion, and changed forecast. These tests reveal sensitivity; they do not prove cause.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Take+one+real+example+and+tweak+it+into+three+made-up+edge+cases%2C+to+see+how+the+model+reacts+to+unusual+inputs.%0Astress%3DX_test.head%281%29.copy%28%29%3B+stress_cases%3D%5B%5D%0Afor+label%2Cchange+in+%5B%28%27unfamiliar+event+type%27%2C%7B%27event_type%27%3A%27new_event_type%27%7D%29%2C%28%27extreme+promotion%27%2C%7B%27social_posts%27%3A999%7D%29%2C%28%27changed+forecast%27%2C%7B%27weather_forecast%27%3A%27storm%27%7D+%29%5D%3A%0A+case%3Dstress.copy%28%29%0A+for+k%2Cv+in+change.items%28%29%3A%0A++if+k+in+case.columns%3A+case%5Bk%5D%3Dv%0A+stress_cases.append%28%7B%27case%27%3Alabel%2C%27prediction%27%3Aint%28chosen.predict%28case%29%5B0%5D%29%7D%29%0Apd.DataFrame%28stress_cases%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Take one real example and tweak it into three made-up edge cases, to see how the model reacts to unusual inputs.
stress=X_test.head(1).copy(); stress_cases=[]
for label,change in [('unfamiliar event type',{'event_type':'new_event_type'}),('extreme promotion',{'social_posts':999}),('changed forecast',{'weather_forecast':'storm'} )]:
 case=stress.copy()
 for k,v in change.items():
  if k in case.columns: case[k]=v
 stress_cases.append({'case':label,'prediction':int(chosen.predict(case)[0])})
pd.DataFrame(stress_cases)

## 9. Propose one improvement and record AI use

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+A+short+note+on+what+could+make+the+model+better+next+time%2C+plus+a+record+of+how+AI+help+was+used+%28for+academic+honesty%29.%0Aimprovement%3D%27Collect+more+examples+for+weak+or+unfamiliar+event+categories%2C+then+re-run+validation+before+touching+a+new+final+test+set.%27%0Aai_use_record%3D%7B%27allowed%27%3A%27AI+explained+code+or+suggested+tests%27%2C%27student_owned%27%3A%27feature+decisions%2C+metric+interpretation%2C+error+analysis%2C+limitations%2C+final+model+card%27%7D"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# A short note on what could make the model better next time, plus a record of how AI help was used (for academic honesty).
improvement='Collect more examples for weak or unfamiliar event categories, then re-run validation before touching a new final test set.'
ai_use_record={'allowed':'AI explained code or suggested tests','student_owned':'feature decisions, metric interpretation, error analysis, limitations, final model card'}

## 10. Export the model card

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&amp;logo=youtube&amp;logoColor=white" alt="Watch the video" style="height:26px; vertical-align:middle;"></a>
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Bundle+everything+about+this+project+%28results%2C+limits%2C+decisions%29+into+one+summary+dict%2C+then+save+it+as+a+JSON+file.%0Amodel_card%3D%7B%27project%27%3A%27Will+They+Show+Up%3F%27%2C%27purpose%27%3A%27Low-stakes+classroom+prediction+of+fictional+campus-event+turnout%27%2C%27blueprint_used%27%3Abp_path.name%2C%27features%27%3Afeatures%2C%27target%27%3Atarget%2C%27selection_evidence%27%3Avalidation%2C%27selected_model%27%3Achosen_name%2C%27final_test_metrics%27%3Ametrics%2C%27confusion_matrix%27%3Acm.tolist%28%29%2C%27baseline%27%3Ablueprint.get%28%27majority_baseline_accuracy%27%29%2C%27known_limits%27%3Ablueprint.get%28%27predicted_failure_cases%27%2C%5B%5D%29%2C%27stress_tests%27%3Astress_cases%2C%27improvement%27%3Aimprovement%2C%27prohibited_uses%27%3Ablueprint.get%28%27prohibited_uses%27%2C%5B%5D%29%2C%27human_review_boundary%27%3Ablueprint.get%28%27human_review_boundary%27%29%2C%27ai_use_record%27%3Aai_use_record%7D%0APath%28%27m9_model_card.json%27%29.write_text%28json.dumps%28model_card%2Cindent%3D2%29%29%3B+print%28%27Saved+m9_model_card.json%27%29%0Amodel_card"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
# Bundle everything about this project (results, limits, decisions) into one summary dict, then save it as a JSON file.
model_card={'project':'Will They Show Up?','purpose':'Low-stakes classroom prediction of fictional campus-event turnout','blueprint_used':bp_path.name,'features':features,'target':target,'selection_evidence':validation,'selected_model':chosen_name,'final_test_metrics':metrics,'confusion_matrix':cm.tolist(),'baseline':blueprint.get('majority_baseline_accuracy'),'known_limits':blueprint.get('predicted_failure_cases',[]),'stress_tests':stress_cases,'improvement':improvement,'prohibited_uses':blueprint.get('prohibited_uses',[]),'human_review_boundary':blueprint.get('human_review_boundary'),'ai_use_record':ai_use_record}
Path('m9_model_card.json').write_text(json.dumps(model_card,indent=2)); print('Saved m9_model_card.json')
model_card